# step6 보강 — Spotlight도 실제로 생성시켜 본다

**어느 스텝:** step6 보강 · **RQ:** RQ 아님(처방 검증) · **무엇을 확인:** 어텐션 증폭(Spotlight)을
건 채로 **실제로 생성된 이름**이 지침을 지키는가

## 왜 필요한가

논문 기여 C3은 두 가지를 한 문장에 담고 있다.

> "잔차 스트림 조향이 어텐션 증폭보다 효과적임을 보이고, 동시에 **선호 점수 회복이 준수율을
> 대변하지 않음**을 밝힌다"

**뒷부분이 앞부분을 무너뜨린다.** 앞부분("값 조향 > Spotlight")은 **선호 점수로만** 재서
나온 결론인데, 뒷부분이 그 점수를 못 믿는다고 선언하기 때문이다.

실제로 `results/step6_steer-generate/`에는 `none`과 `value_add`뿐이고
**`attn_amplify`는 0개다.** Spotlight는 실제 생성을 한 번도 안 돌렸다.

## "점수가 낮으니 준수율도 낮겠지"는 못 쓴다

한때 그렇게 넘어가려 했으나 데이터가 반박한다. 짝지은 20개 조건에서:

| 조건 | 점수 회복률 | 실제 준수율 |
|---|---|---|
| DeepSeek 세기1 | **0.462** (천장의 절반도 안 됨) | **0.857** |
| StableCode 세기1 | **0.433** (거의 같다) | **0.000** |

**거의 같은 점수에서 준수율이 정반대다.** 점수가 높든 낮든 실제 행동을 말해 주지 않는다
(→ `docs/step6/results.md` §3-3).

## 무엇을 바꾸나

기존 생성 검증과 **완전히 같은 절차**이고, 처방만 값 조향 → Spotlight로 바꾼다.

| | 기존 (이미 있다) | 이번 |
|---|---|---|
| 출발점 | 절벽 바닥(지침 camel, 앞 코드 12개 전부 snake) | **동일** |
| 처방 | 값 조향 (층 1개 × 세기 4종) | **Spotlight (스팬 2종 × ψ 2종)** |
| 재는 것 | 생성된 이름의 준수 여부 + 이름 건전성 | **동일** |
| 묶음 | 21개 | **동일** |

**무개입은 다시 안 돌린다** — 이미 84개 있고 하한선으로 쓴다.

## 결과가 어느 쪽으로 나오든 무슨 뜻인지 (미리 밝힌다)

| 나올 수 있는 그림 | 뜻 | C3에 미치는 영향 |
|---|---|---|
| Spotlight 준수율 ≈ 0 | 점수가 0 근처였던 것과 일치 | **C3가 실제 준수율로 선다.** 예상하는 결과 |
| Spotlight 준수율이 값 조향에 필적 | **점수가 Spotlight를 과소평가했다** | **C3 앞부분을 철회하거나 크게 좁혀야 한다** |
| Llama만 작동 | 점수에서 본 것(0.904)의 확인 | C3를 "코드 특화 모델에 한정"으로 명시 |
| Spotlight 이름이 깨짐 | 어텐션 증폭도 과하면 망가진다 | 새 관찰(부작용 대칭성) |

**예상:** Qwen·DeepSeek·StableCode는 0에 가깝고 Llama만 어느 정도 나올 것으로 본다.
**다르게 나오면 조건을 바꾸지 않고 그대로 기록한다**(CLAUDE.md §4).

## 부하

조건 **84개/모델**(21묶음 × 4조건). 생성이라 점수보다 느리지만 24토큰까지만 만든다.
Spotlight는 **전 층 어텐션 함수를 갈아끼우므로** 값 조향보다 토큰당 조금 더 느리다.
T4에서 모델 하나에 **20~40분** 예상. DeepSeek-6.7B는 메모리가 빠듯하니 다른 노트북을
동시에 켜 두지 말 것.

In [ ]:
# ② 환경 · 시드 42
!pip install -q -r requirements.txt
import random, numpy as np, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (매우 느림)')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
# ③ 저장소 · 브랜치
import os
if not os.path.isdir('HCLT_2026'):
    !git clone https://github.com/deanjs/HCLT_2026.git
%cd HCLT_2026
BRANCH = 'integration/step1-5'
!git fetch --quiet origin $BRANCH
!git checkout $BRANCH
!git pull --quiet origin $BRANCH
!pip install -e . -q
import sys; sys.path.insert(0, 'src')
print('브랜치:', BRANCH)

In [ ]:
# ④ 조건 설정
from harness.conditions import (Condition, ModelSpec, PrecedingCode, Instruction,
                                Composition, InstructionForm, Notation,
                                Intervention, InterventionKind)

MODELS = [
    ModelSpec(name='Qwen/Qwen2.5-Coder-3B-Instruct',           family='qwen',      dtype='float16'),
    ModelSpec(name='deepseek-ai/deepseek-coder-6.7b-instruct',  family='deepseek',  dtype='float16'),
    ModelSpec(name='unsloth/Llama-3.2-3B-Instruct',             family='llama',     dtype='float16'),
    ModelSpec(name='stabilityai/stable-code-instruct-3b',       family='stability', dtype='float16'),
]

# ★ 이번에 돌릴 모델 (0=qwen, 1=deepseek, 2=llama, 3=stable)
PICK = 0
MODEL = MODELS[PICK]
print('이번 모델:', MODEL.family)

# 본실험(step6_steer)의 Spotlight 조건과 **똑같이** 맞춘다. 여기가 어긋나면 짝이 안 맞는다.
BLOCKS = list(range(21))                       # 기존 생성 검증과 같은 묶음
SPANS  = ['rule_word', 'instruction']          # 규칙문 지시어만 / 지침 문장 전체
PSIS   = [0.1, 0.3]                            # 원문 기본값 / 강한 조건

INSTR = Instruction(form=InstructionForm.POSITIVE, target_notation=Notation.CAMEL)

def base(block):
    return dict(model=MODEL,
                preceding=PrecedingCode(n_compliant=0, n_functions=12,
                                        composition=Composition.POOL, pool_block=block),
                instruction=INSTR, seed=SEED, tag='cliff')

spot_conditions = []
for b in BLOCKS:
    for span in SPANS:
        for psi in PSIS:
            spot_conditions.append(Condition(**base(b), intervention=Intervention(
                kind=InterventionKind.ATTENTION_AMPLIFY, layers='all',
                amplify=psi, span=span)))

print(f'Spotlight 생성 조건 {len(spot_conditions)}개  '
      f'(묶음 {len(BLOCKS)} x 스팬 {len(SPANS)} x psi {len(PSIS)})')
assert len({c.slug() for c in spot_conditions}) == len(spot_conditions), '슬러그 충돌'

# 본실험에 같은 슬러그가 있는지 확인 — 있어야 정상이다(같은 조건을 점수로 이미 쟀다).
from harness import result_path
have_score = sum(result_path(c, step='step6_steer').exists() for c in spot_conditions)
print(f'같은 조건의 점수 결과가 이미 있는 것: {have_score}/{len(spot_conditions)}개')
if have_score != len(spot_conditions):
    print('  ⚠️ 점수 결과가 없는 조건이 있다. 짝지어 비교하려면 step6_steer를 먼저 받아 둘 것.')

In [ ]:
# ⑤ 실행 — 이미 있는 조건은 건너뛴다(재개)
from harness import run, ResultRecord, save_result, result_path
from harness.model import load_model
import numpy as np
from collections import defaultdict

STEP = 'step6_steer-generate'          # 값 조향 생성분과 **같은 폴더**. 슬러그가 방법을 구분한다
todo = [c for c in spot_conditions if not result_path(c, step=STEP).exists()]
print(f'[{MODEL.family}] 전체 {len(spot_conditions)}개 중 남은 조건 {len(todo)}개')

if todo:
    handle = load_model(MODEL)
    seen = defaultdict(list)
    for i, c in enumerate(todo, 1):
        out = run(c, handle=handle, mode='steer_generate', max_new_tokens=24)
        save_result(ResultRecord(condition=out.condition, metrics=out.metrics,
                                 step=STEP, rq='RQ2/RQ3'))
        ex = out.metrics.extra
        seen[f"{ex['span']} psi{ex['psi_target']:g}"].append((ex['compliant'], ex['name_ok'], ex['name']))
        del out
        if i % 20 == 0 or i == len(todo):
            parts = []
            for k in sorted(seen):
                v = seen[k]
                parts.append(f"{k} 지킴{np.mean([a for a,_,_ in v]):.2f}"
                             f"/멀쩡{np.mean([b for _,b,_ in v]):.2f}")
            print(f'  [{i}/{len(todo)}] ' + ' | '.join(parts))
            print(f'          최근 나온 이름: {[n for *_ , n in list(seen.values())[-1][-3:]]}')
    del handle
    import torch, gc; gc.collect(); torch.cuda.empty_cache()
print('완료 — 결과는 results/' + STEP + '/ 에 저장됐다')

In [ ]:
# ⑥ 요약 — Spotlight가 실제로 이름을 바꿨나
import numpy as np
from collections import defaultdict
from harness.results import load_result
from harness import result_path

recs = [load_result(result_path(c, step='step6_steer-generate')) for c in spot_conditions
        if result_path(c, step='step6_steer-generate').exists()]
by = defaultdict(list)
for r in recs:
    ex = r.metrics.extra
    by[f"{ex['span']} psi{ex['psi_target']:g}"].append(ex)

print(f'[{MODEL.family}] Spotlight — 조향을 건 채로 실제 생성한 이름\n')
print(f"{'조건':>22}{'지침 지킴':>10}{'멀쩡한 이름':>12}{'camel':>8}{'snake':>8}{'그 외':>8}{'개수':>6}")
for k in sorted(by):
    v = by[k]
    nota = [e['notation'] for e in v]
    print(f"{k:>22}{np.mean([e['compliant'] for e in v]):>10.3f}"
          f"{np.mean([e['name_ok'] for e in v]):>12.3f}"
          f"{nota.count('camel')/len(v):>8.2f}{nota.count('snake')/len(v):>8.2f}"
          f"{nota.count('other')/len(v):>8.2f}{len(v):>6}")

print('\n실제로 나온 이름 (조건마다 3개씩):')
for k in sorted(by):
    print(f'  {k}: {[e["name"] for e in by[k]][:3]}')

print('\n읽는 법')
print('  「지침 지킴」이 0에 가까우면  → 어텐션을 키워도 행동이 안 바뀐다 (예상한 결과)')
print('  「지침 지킴」이 0.8을 넘으면  → 점수가 Spotlight를 과소평가했다. C3를 고쳐야 한다')
print('  「멀쩡한 이름」이 낮으면      → 어텐션 증폭도 과하면 이름을 망가뜨린다 (새 관찰)')

In [ ]:
# ⑦ 무개입·값 조향과 나란히 — 이 노트북의 결론
import numpy as np, glob, json
from collections import defaultdict

rows = defaultdict(list)
for p in glob.glob('results/step6_steer-generate/*.json'):
    r = json.loads(open(p, encoding='utf-8').read())
    if r['condition']['model']['family'] != MODEL.family:
        continue
    ex = r['metrics']['extra']
    if ex['method'] == 'none':
        key = '무개입'
    elif ex['method'] == 'value_add':
        key = f"값 조향 세기{ex['strength']:g}"
    else:
        key = f"Spotlight {ex['span']} psi{ex['psi_target']:g}"
    rows[key].append(ex)

print(f'[{MODEL.family}] 실제 생성 준수율 — 처방별\n')
print(f"{'처방':>28}{'지침 지킴':>10}{'멀쩡한 이름':>12}{'개수':>6}")
for k in sorted(rows, key=lambda x: (x != '무개입', x)):
    v = rows[k]
    print(f"{k:>28}{np.mean([e['compliant'] for e in v]):>10.3f}"
          f"{np.mean([e['name_ok'] for e in v]):>12.3f}{len(v):>6}")

print('\n※ 값 조향과 Spotlight는 조건 축이 다르다(세기 vs psi). 크기를 직접 비교하지 말고')
print('   「무개입(하한)에서 얼마나 올라갔나」로 읽는다.')

In [ ]:
# ⑧ 이번에 만든 것만 zip으로 묶어 내려받기
import shutil, os, glob
from harness import result_path

# ⚠️ results/step6_steer-generate/ 에는 **이미 420개(4모델 값 조향분)** 가 들어 있다.
#    폴더를 통째로 묶으면 이번에 안 돌린 것까지 딸려 나오고, 그걸 다시 풀어 커밋하면
#    불변이어야 할 기존 결과를 덮어쓴다(CLAUDE.md §6). 그래서 **이번 조건만** 골라 담는다.
STEP = 'step6_steer-generate'
made = [result_path(c, step=STEP) for c in spot_conditions
        if result_path(c, step=STEP).exists()]
print(f'{STEP} 폴더 전체: {len(glob.glob(f"results/{STEP}/*.json"))}개 '
      f'(대부분 저장소에 원래 있던 값 조향분)')
print(f'이번에 만든 Spotlight 결과({MODEL.family}): {len(made)}개')

if not made:
    print('\n담을 것이 없다 — 실행 셀(⑤)을 먼저 돌릴 것')
else:
    stage = f'upload_{STEP}_{MODEL.family}'
    shutil.rmtree(stage, ignore_errors=True); os.makedirs(stage)
    for f in made:
        shutil.copy(f, stage)
    path = shutil.make_archive(stage, 'zip', stage)
    print(f'\n{path} ({os.path.getsize(path)/1e6:.1f}MB, {len(made)}개)')
    try:
        from google.colab import files
        files.download(path)
        print('다운로드 시작. 브라우저가 막으면 왼쪽 파일 탐색기에서 직접 받으면 된다.')
    except Exception as e:
        print('자동 다운로드 실패:', e, '— 왼쪽 파일 탐색기에서 직접 받을 것')

print('\n받은 zip은 통합 브랜치의 results/step6_steer-generate/ 에 **풀어 넣기만** 한다')
print('(같은 이름 파일이 없어야 정상 — 있으면 이미 돌린 조건이다).')
print('네 모델을 다 모은 뒤:  python scripts/step6_summary.py')